In [ ]:
# === 노트북 공통 preamble (llm_math 패키지 로드) ===
import sys
from pathlib import Path

# llm_math 패키지를 찾을 수 있는 후보 경로들
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 현재 디렉토리의 상위 디렉토리들도 후보에 추가 (notebooks/ 폴더에서 실행 시)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math import 시도
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[주의] llm_math 패키지 로드 실패: {e}")
    print("  GitHub 레포를 클론하고 colab_setup.sh를 실행하세요.")
# === preamble 끝 ===


# Ch 31. nano-GPT — 처음부터 끝까지 ⭐

> **학습 목표**
> - 토크나이저 → 모델 → 학습 → 생성의 전 과정을 직접 구현한다
> - Multi-Head Attention + RoPE + SwiGLU FFN 모듈 조립
> - 사전학습 + SFT 단계 수행
> - CPU vs GPU에서 학습 속도를 최종 비교한다

이 장은 앞서 배운 모든 것을 모아 Karpathy 스타일 nano-GPT를 직접 만듭니다.

## 31.1 프로젝트 개요와 아키텍처

만들 것:
1. **문자 단위 BPE 토크나이저** (간소화)
2. **GPT 모델**: RoPE 위치 인코딩, GQA 어텐션, SwiGLU FFN, RMSNorm
3. **사전학습**: Tiny Shakespeare로 다음 토큰 예측
4. **생성**: 다양한 디코딩 전략으로 텍스트 생성
5. **벤치마크**: CPU vs GPU 학습 속도

## 31.2 데이터 준비


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time
from llm_math.data import load_tiny_shakespeare

# Tiny Shakespeare 로드
text = load_tiny_shakespeare(max_chars=50000)
chars = sorted(set(text))
vocab_size = len(chars)
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}
data = np.array([char_to_idx[c] for c in text])
print(f"텍스트 Length: {len(text):,}")
print(f"Vocabulary Size: {vocab_size}")
print(f"처음 200자: {text[:200]}")

# 학습/검증 분할
n_train = int(len(data) * 0.9)
train_data = data[:n_train]
val_data = data[n_train:]
print(f"\nTraining: {len(train_data):,}, Validation: {len(val_data):,}")

def get_batch(split, seq_len, batch_size):
    source = train_data if split == 'train' else val_data
    starts = np.random.randint(0, len(source) - seq_len - 1, batch_size)
    x = np.stack([source[s:s+seq_len] for s in starts])
    y = np.stack([source[s+1:s+1+seq_len] for s in starts])
    return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)


## 31.3 nano-GPT 모델 구현

핵심 모듈:
- **RoPE** (위치 인코딩)
- **GQA** (Grouped Query Attention)
- **SwiGLU** (FFN)
- **RMSNorm** (정규화)
- **Pre-LN** (잔차)


In [ ]:
# RoPE 구현
def rotate_half(x):
    x1, x2 = x[..., :x.shape[-1] // 2], x[..., x.shape[-1] // 2:]
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(x, theta_base=10000.0):
    """x: (B, H, T, d)."""
    B, H, T, d = x.shape
    freqs = 1.0 / (theta_base ** (torch.arange(0, d, 2).float() / d))
    angles = torch.arange(T, device=x.device).float()[:, None] * freqs[None, :]  # (T, d/2)
    cos = torch.cos(angles).repeat_interleave(2, dim=-1)  # (T, d)
    sin = torch.sin(angles).repeat_interleave(2, dim=-1)
    cos = cos[None, None, :, :]  # (1, 1, T, d)
    sin = sin[None, None, :, :]
    return x * cos + rotate_half(x) * sin

# RMSNorm
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(dim))
        self.eps = eps
    def forward(self, x):
        rms = torch.sqrt((x ** 2).mean(dim=-1, keepdim=True) + self.eps)
        return x / rms * self.gamma

# GQA Attention + RoPE
class GQAAttention(nn.Module):
    def __init__(self, d_model, n_q_heads, n_kv_heads):
        super().__init__()
        self.n_q_heads = n_q_heads
        self.n_kv_heads = n_kv_heads
        self.d_k = d_model // n_q_heads
        self.n_rep = n_q_heads // n_kv_heads
        self.W_q = nn.Linear(d_model, n_q_heads * self.d_k, bias=False)
        self.W_k = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.W_v = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.W_o = nn.Linear(n_q_heads * self.d_k, d_model, bias=False)

    def forward(self, x, mask):
        B, T, D = x.shape
        q = self.W_q(x).view(B, T, self.n_q_heads, self.d_k).transpose(1, 2)
        k = self.W_k(x).view(B, T, self.n_kv_heads, self.d_k).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.n_kv_heads, self.d_k).transpose(1, 2)
        # RoPE Application
        q = apply_rope(q)
        k = apply_rope(k)
        # KV를 Q Head 수만큼 Iteration
        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)
        scores = q @ k.transpose(-1, -2) / (self.d_k ** 0.5) + mask
        weights = F.softmax(scores, dim=-1)
        out = (weights @ v).transpose(1, 2).contiguous().view(B, T, -1)
        return self.W_o(out)

# SwiGLU FFN
class SwiGLUFFN(nn.Module):
    def __init__(self, d, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d, bias=False)
        self.w3 = nn.Linear(d, d_ff, bias=False)
    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

# Transformer Block
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_q_heads, n_kv_heads, d_ff):
        super().__init__()
        self.attn = GQAAttention(d_model, n_q_heads, n_kv_heads)
        self.ffn = SwiGLUFFN(d_model, d_ff)
        self.norm1 = RMSNorm(d_model)
        self.norm2 = RMSNorm(d_model)

    def forward(self, x, mask):
        x = x + self.attn(self.norm1(x), mask)
        x = x + self.ffn(self.norm2(x))
        return x

# nano-GPT
class NanoGPT(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_q_heads=4, n_kv_heads=2,
                 n_layers=4, d_ff=512, max_seq_len=256):
        super().__init__()
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_q_heads, n_kv_heads, d_ff)
            for _ in range(n_layers)
        ])
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        # weight tying
        self.lm_head.weight = self.token_emb.weight

    def forward(self, x):
        B, T = x.shape
        emb = self.token_emb(x) * (self.d_model ** 0.5)
        mask = torch.triu(torch.full((T, T), float('-inf'), device=x.device), diagonal=1)
        h = emb
        for block in self.blocks:
            h = block(h, mask)
        h = self.norm_f(h)
        return self.lm_head(h)

# Model Generation
torch.manual_seed(42)
model = NanoGPT(vocab_size, d_model=128, n_q_heads=4, n_kv_heads=2,
                n_layers=4, d_ff=512, max_seq_len=256)
n_params = sum(p.numel() for p in model.parameters())
print(f"nano-GPT 파라미터 수: {n_params:,}")
print(f"Composition: d=128, L=4, H=4, KV=2 (GQA), FFN=512 (SwiGLU), RoPE")


## 31.4 학습 스크립트


In [ ]:
# 사전학습
from llm_math.bench import time_fn

opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01, betas=(0.9, 0.95))
loss_fn = nn.CrossEntropyLoss()

# 학습률 스케줄러 (warmup + cosine)
def get_lr(step, max_steps, warmup=50, lr_max=3e-4, lr_min=3e-5):
    if step < warmup:
        return lr_max * step / warmup
    progress = (step - warmup) / (max_steps - warmup)
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * progress))

n_steps = 300
seq_len = 128
batch_size = 32

train_losses = []
val_losses = []
t0 = time.time()
for step in range(n_steps):
    # 학습률 업데이트
    lr = get_lr(step, n_steps)
    for g in opt.param_groups:
        g['lr'] = lr

    model.train()
    x, y = get_batch('train', seq_len, batch_size)
    opt.zero_grad()
    logits = model(x)
    loss = loss_fn(logits.reshape(-1, vocab_size), y.reshape(-1))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    train_losses.append(loss.item())

    # 주기적 Validation
    if (step + 1) % 50 == 0:
        model.eval()
        with torch.no_grad():
            vx, vy = get_batch('val', seq_len, batch_size)
            vlogits = model(vx)
            vloss = loss_fn(vlogits.reshape(-1, vocab_size), vy.reshape(-1))
            val_losses.append(vloss.item())
        print(f"스텝 {step+1}: train_loss={loss.item():.4f}, val_loss={vloss.item():.4f}, lr={lr:.2e}")

print(f"\nTraining Time: {time.time() - t0:.1f}초")
print(f"최종 train loss: {train_losses[-1]:.4f} (PPL={np.exp(train_losses[-1]):.2f})")


In [ ]:
# 학습 곡선
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(train_losses, alpha=0.3, color='blue', label='train')
window = 20
smoothed = np.convolve(train_losses, np.ones(window)/window, mode='valid')
axes[0].plot(range(window-1, len(train_losses)), smoothed, 'r-', linewidth=2)
axes[0].set_xlabel('스텝'); axes[0].set_ylabel('Loss')
axes[0].set_title('nano-GPT Learning Curve')
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(50, n_steps+1, 50), val_losses, 'go-', label='val')
axes[1].set_xlabel('스텝'); axes[1].set_ylabel('Loss')
axes[1].set_title(f'Validation Loss (최종: {val_losses[-1]:.4f})')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch31_nanogpt_training.png', dpi=100, bbox_inches='tight')
plt.show()


## 31.5 텍스트 생성


In [ ]:
# 텍스트 생성 (여러 디코딩 전략)
def generate(model, prompt, n_new=200, temperature=0.8, top_k=None, top_p=None):
    model.eval()
    idx = [char_to_idx[c] for c in prompt if c in char_to_idx]
    with torch.no_grad():
        for _ in range(n_new):
            x = torch.tensor([idx[-256:]], dtype=torch.long)
            logits = model(x)
            logit = logits[0, -1] / max(temperature, 1e-8)
            # top-k
            if top_k is not None:
                v, _ = torch.topk(logit, top_k)
                logit[logit < v[-1]] = -float('inf')
            # top-p
            if top_p is not None:
                sorted_logits, sorted_idx = torch.sort(logit, descending=True)
                cumprobs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                mask = cumprobs > top_p
                mask[1:] = mask[:-1].clone()
                mask[0] = False
                sorted_logits[mask] = -float('inf')
                logit = sorted_logits[torch.argsort(sorted_idx)]
            probs = F.softmax(logit, dim=-1)
            next_idx = torch.multinomial(probs, 1).item()
            idx.append(next_idx)
    return ''.join(idx_to_char.get(i, '?') for i in idx)

# 다양한 디코딩 비교
print("=" * 60)
print("1. Temperature=0.7, Top-k=40")
print("=" * 60)
print(generate(model, "ROMEO:\n", n_new=200, temperature=0.7, top_k=40))
print()
print("=" * 60)
print("2. Temperature=1.0, Top-p=0.9")
print("=" * 60)
print(generate(model, "To be, or", n_new=200, temperature=1.0, top_p=0.9))


## 31.6 [CPU/GPU 벤치마크 ⑬] 학습 속도 최종 비교


In [ ]:
# CPU vs GPU 학습 속도 비교
def benchmark_training(model_fn, n_warmup=5, n_steps=20, device='cpu'):
    model = model_fn().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
    loss_fn = nn.CrossEntropyLoss()

    # warmup
    for _ in range(n_warmup):
        x, y = get_batch('train', 128, 32)
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        logits = model(x)
        loss = loss_fn(logits.reshape(-1, vocab_size), y.reshape(-1))
        loss.backward()
        opt.step()

    # Measurement
    if device == 'cuda':
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(n_steps):
        x, y = get_batch('train', 128, 32)
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        logits = model(x)
        loss = loss_fn(logits.reshape(-1, vocab_size), y.reshape(-1))
        loss.backward()
        opt.step()
    if device == 'cuda':
        torch.cuda.synchronize()
    elapsed = time.time() - t0
    return elapsed / n_steps * 1000  # ms/step

# CPU
torch.manual_seed(0)
t_cpu = benchmark_training(lambda: NanoGPT(vocab_size, d_model=128, n_q_heads=4,
                                            n_kv_heads=2, n_layers=4, d_ff=512),
                           device='cpu')
print(f"CPU: {t_cpu:.2f} ms/step ({1000/t_cpu:.1f} steps/sec)")

if torch.cuda.is_available():
    torch.manual_seed(0)
    t_gpu = benchmark_training(lambda: NanoGPT(vocab_size, d_model=128, n_q_heads=4,
                                                n_kv_heads=2, n_layers=4, d_ff=512),
                               device='cuda')
    print(f"GPU: {t_gpu:.2f} ms/step ({1000/t_gpu:.1f} steps/sec)")
    print(f"Speed Improvement: {t_cpu/t_gpu:.2f}x")
else:
    print("\nGPU 없음. Colab T4 GPU로 전환하면 5-10x 가속 기대.")

print(f"\n500 스텝 Training Time Estimation:")
print(f"  CPU: {t_cpu * 500 / 1000:.1f}초")
if torch.cuda.is_available():
    print(f"  GPU: {t_gpu * 500 / 1000:.1f}초")


## 31.7 모델 저장 및 로드


In [ ]:
# 모델 저장
import os
os.makedirs('../checkpoints', exist_ok=True)
torch.save({
    'model_state_dict': model.state_dict(),
    'config': {
        'vocab_size': vocab_size,
        'd_model': 128,
        'n_q_heads': 4,
        'n_kv_heads': 2,
        'n_layers': 4,
        'd_ff': 512,
    }
}, '../checkpoints/nanogpt_shakespeare.pt')
print("Model 저장 완료: ../checkpoints/nanogpt_shakespeare.pt")
print(f"파일 Magnitude: {os.path.getsize('../checkpoints/nanogpt_shakespeare.pt') / 1024:.1f} KB")

# 로드 Test
ckpt = torch.load('../checkpoints/nanogpt_shakespeare.pt', weights_only=False)
new_model = NanoGPT(**ckpt['config'])
new_model.load_state_dict(ckpt['model_state_dict'])
print("Model 로드 성공!")


## 31.8 핵심 정리

이 장에서 만든 것:
- ✅ RoPE 위치 인코딩
- ✅ GQA 어텐션
- ✅ SwiGLU FFN
- ✅ RMSNorm
- ✅ Weight Tying
- ✅ Warmup + Cosine LR
- ✅ 그래디언트 Clipping
- ✅ Top-k, Top-p 디코딩

이것은 LLaMA/GPT 구조의 핵심 요소를 모두 포함한 "진짜" LLM입니다.
확장하려면: 더 큰 모델, 더 많은 데이터, 더 긴 학습.

## 연습문제

1. d_model=256, n_layers=8로 모델을 키우고 학습하라.
2. RoPE 대신 학습된 위치 임베딩을 사용하고 결과를 비교하라.
3. SwiGLU 대신 표준 FFN을 사용하고 성능을 비교하라.
4. 학습 스텝 수 500, 1000, 2000에 따른 생성 품질을 비교하라.
5. 다른 데이터셋(한국어 텍스트)로 학습해 보라.

> 해설: `solutions/ch31_solutions.ipynb`
